# Black Pearl Restaurant Data Analysis

This notebook analyzes a manually compiled dataset of 233 restaurants listed in the Black Pearl Restaurant Guide. It rebuilds all derived tables and figures from the preserved source workbook. Run the cells in order; committed outputs are intentionally cleared so they cannot be mistaken for executed results.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import run_analysis

# Rebuild every result and figure from the source workbook.
artifacts = run_analysis(ROOT, build_notebook=False)
data = pd.read_csv(ROOT / 'data/public/black_pearl_restaurants.csv')
print(f'Loaded {len(data)} restaurants and {len(data.columns)} public fields.')

## 1. Data audit

The source workbook contains 233 restaurant records. Its other worksheets contain historical Excel outputs; one regression worksheet used only the first 153 records. The analysis below uses all 233 records. The public dataset removes the redundant `hello` field and retains source labels alongside normalized geography fields.

In [ ]:
quality = pd.read_csv(ROOT / 'results/data_quality_summary.csv')
quality

## 2. Descriptive results

Summary statistics are calculated from the 233-row analysis sample.

![Score distributions](../figures/score_distributions.png)

In [ ]:
descriptive = pd.read_csv(ROOT / 'results/descriptive_statistics.csv')
descriptive

## 3. Sample-version reconciliation

This quality-control table documents the difference between the historical 153-row Excel output and the final 233-row dataset.

In [ ]:
comparison = pd.read_csv(ROOT / 'results/regression_sample_comparison.csv')
comparison.round(6)

## 4. Core OLS analysis

The response is the public platform total score. Predictors are the public taste, environment, and service subscores. The model describes association among public rating fields; it does **not** reveal confidential Black Pearl jury selection weights.

In [ ]:
regression = pd.read_csv(ROOT / 'results/core_regression.csv')
regression.round(6)

The fitted equation is:

$$\widehat{TotalScore}=0.1817+0.6174(Taste)+0.1963(Environment)+0.1436(Service)$$

Model $R^2=0.8925$ and adjusted $R^2=0.8911$. These slopes are unstandardized; standardized coefficients are reported separately.

![Regression coefficients](../figures/regression_coefficients.png)

In [ ]:
betas = pd.read_csv(ROOT / 'results/standardized_betas.csv')
vif = pd.read_csv(ROOT / 'results/vif.csv')
display(betas.round(6))
display(vif.round(4))

The VIF values are below 5, although service has the highest shared variance. Because the outcome and predictors are rounded platform ratings, diagnostics should be interpreted cautiously. HC3 robust standard errors are included in the result table.

![Residual diagnostics](../figures/residual_diagnostics.png)

## 5. Award-count analysis

In [ ]:
awards = pd.read_csv(ROOT / 'results/award_tests.csv')
awards.round(6)

The paired Wilcoxon test is significant and the counts are strongly associated (Spearman $\rho=0.709$). Paired $d_z$ is about 0.295 in magnitude.

![Award relationship](../figures/award_relationship.png)

## 6. Cuisine and region

In [ ]:
cuisine_region = pd.read_csv(ROOT / 'results/cuisine_region_test.csv')
cuisine_region.round(6)

Cramér's V is about 0.616, but 97.1% of expected cells are below 5. The asymptotic chi-square p-value should not be trusted on its own, so the analysis adds a fixed-seed permutation test. With zero exceedances in 5,000 draws, $1/5001$ is the Monte Carlo resolution bound, not an exact population p-value.

## 7. Regional price differences and diamond-level limitation

In [ ]:
regional = pd.read_csv(ROOT / 'results/region_price_omnibus.csv')
diamond = pd.read_csv(ROOT / 'results/diamond_summary.csv')
display(regional.round(6))
display(diamond.round(4))

Only five observations are three-diamond restaurants, so a three-class prediction model would be unstable. A simple one-way ANOVA gives $p=0.846$.

![Regional prices](../figures/price_by_region.png)

## Conclusion

Taste has the largest conditional association with the public total rating among the three component scores in this sample. This is descriptive evidence, not a causal claim or a reconstruction of the Black Pearl selection algorithm. The diamond-level sample is too imbalanced for a reliable multiclass prediction model.